# Notebook 19 — MS-ASL Downloader & Trimmer

Descarga y recorta los clips del dataset **MS-ASL** (Microsoft, 1000 clases, ~25K clips).

## Estrategia de descarga

Los JSONs de MS-ASL tienen hasta **194 clips distintos apuntando al mismo vídeo de YouTube**.
En lugar de descargar el mismo vídeo N veces, agrupamos todos los clips por URL:

```
Para cada URL única:
  1. Descargar el vídeo completo a /tmp (una sola vez)
  2. Para cada clip de esa URL → ffmpeg trim + crop + resize
  3. Borrar el vídeo temporal
```

## Output
```
06_datasets/MS-ASL/videos/
├── train/{label:04d}/{clip_id}.mp4
├── val/{label:04d}/{clip_id}.mp4
├── test/{label:04d}/{clip_id}.mp4
├── _progress.json        ← resume automático
├── _errors.json          ← errores clasificados
├── _audit.json           ← resumen final
└── missing_videos.log    ← log legible para el tutor
```

## Siguiente paso
→ **Notebook 20**: extracción de landmarks MediaPipe (mismo pipeline que Nb 15 de WLASL)

In [9]:
# ── Dependencias ─────────────────────────────────────────────────────────────
import subprocess, sys

def ensure(pkg, import_as=None):
    name = import_as or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

ensure('yt_dlp', 'yt_dlp')
ensure('tqdm')

import json, os, hashlib, shutil, logging, time
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta
import yt_dlp
from tqdm.notebook import tqdm

print('yt-dlp:', yt_dlp.version.__version__)
print('ffmpeg:', subprocess.check_output(['ffmpeg', '-version']).decode().split('\n')[0])

yt-dlp: 2026.03.03
ffmpeg: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [10]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
MSASL_DIR    = Path('../../06_datasets/MS-ASL')
OUTPUT_DIR   = Path('../../06_datasets/MS-ASL/videos')
TEMP_DIR     = Path('/tmp/msasl_temp')

PROGRESS_FILE = OUTPUT_DIR / '_progress.json'
ERRORS_FILE   = OUTPUT_DIR / '_errors.json'
AUDIT_FILE    = OUTPUT_DIR / '_audit.json'
LOG_FILE      = OUTPUT_DIR / 'missing_videos.log'

# Vídeo: resolución máxima de descarga (480p es suficiente para landmarks)
MAX_HEIGHT = 480

# Crop al signante
CROP_SIGNER = True    # Recortar al bounding box del signante
CROP_MARGIN = 0.12    # Margen extra alrededor del box (fracción del lado)
TARGET_HEIGHT = 360   # Altura final del clip (-1 = no redimensionar)

# Control
CHECKPOINT_EVERY = 20   # Guardar progreso cada N URLs procesadas
MAX_RETRIES = 1         # Reintentos por URL en caso de error de red

# Crear directorios
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUTPUT_DIR.resolve())
print('Temp dir:  ', TEMP_DIR.resolve())

Output dir: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/MS-ASL/videos
Temp dir:   /tmp/msasl_temp


In [11]:
# ── Cargar JSONs y construir manifiesto agrupado por URL ──────────────────────
with open(MSASL_DIR / 'MSASL_train.json') as f: train_data = json.load(f)
with open(MSASL_DIR / 'MSASL_val.json')   as f: val_data   = json.load(f)
with open(MSASL_DIR / 'MSASL_test.json')  as f: test_data  = json.load(f)
with open(MSASL_DIR / 'MSASL_classes.json') as f: classes = json.load(f)

def make_clip_id(url: str, start: int, end: int) -> str:
    """ID único para un clip: hash de (url, start_frame, end_frame)."""
    key = f"{url}_{start}_{end}"
    return hashlib.sha1(key.encode()).hexdigest()[:12]

# Manifiesto: url → lista de clips
manifest: dict = defaultdict(list)

for split_name, entries in [('train', train_data), ('val', val_data), ('test', test_data)]:
    for entry in entries:
        clip_id = make_clip_id(entry['url'], entry['start'], entry['end'])
        label   = entry['label']
        out_path = OUTPUT_DIR / split_name / f"{label:04d}" / f"{clip_id}.mp4"
        manifest[entry['url']].append({
            'clip_id'    : clip_id,
            'split'      : split_name,
            'label'      : label,
            'text'       : entry.get('text', ''),
            'start_time' : entry['start_time'],
            'end_time'   : entry['end_time'],
            'start_frame': entry['start'],
            'end_frame'  : entry['end'],
            'box'        : entry['box'],          # [y0, x0, y1, x1] normalized
            'orig_width' : entry.get('width', 0),
            'orig_height': entry.get('height', 0),
            'out_path'   : str(out_path),
        })

print(f'Splits cargados: train={len(train_data)}  val={len(val_data)}  test={len(test_data)}')
print(f'URLs únicas: {len(manifest):,}')
print(f'Clips totales en manifiesto: {sum(len(v) for v in manifest.values()):,}')

Splits cargados: train=16054  val=5287  test=4172
URLs únicas: 7,213
Clips totales en manifiesto: 25,513


In [12]:
# ── Estadísticas del manifiesto ───────────────────────────────────────────────
clip_counts = sorted([len(v) for v in manifest.values()])
total_clips = sum(clip_counts)

durations_all = []
for clips in manifest.values():
    for c in clips:
        durations_all.append(c['end_time'] - c['start_time'])

print('── Distribución clips por URL ──────────────────────')
print(f'  min={min(clip_counts)}  median={clip_counts[len(clip_counts)//2]}  '
      f'mean={sum(clip_counts)/len(clip_counts):.1f}  max={max(clip_counts)}')
urls_multi = sum(1 for c in clip_counts if c > 1)
print(f'  URLs con >1 clip: {urls_multi} ({100*urls_multi/len(clip_counts):.1f}%)')
print(f'  Clips totales:    {total_clips:,}')

print()
print('── Duración de clips (segundos) ────────────────────')
print(f'  min={min(durations_all):.2f}s  mean={sum(durations_all)/len(durations_all):.2f}s  '
      f'max={max(durations_all):.2f}s')
total_h = sum(durations_all) / 3600
print(f'  Duración total de clips: {total_h:.1f} horas')

print()
print('── Por split ────────────────────────────────────────')
for split_name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    labels_in_split = set(e['label'] for e in data)
    print(f'  {split_name:5s}: {len(data):5d} clips  {len(labels_in_split):4d} clases')

print()
print('── Eficiencia de agrupación ─────────────────────────')
saved_downloads = total_clips - len(manifest)
print(f'  Descargas únicas:  {len(manifest):,}')
print(f'  Descargas evitadas: {saved_downloads:,} ({100*saved_downloads/total_clips:.1f}%)')

── Distribución clips por URL ──────────────────────
  min=1  median=1  mean=3.5  max=194
  URLs con >1 clip: 1121 (15.5%)
  Clips totales:    25,513

── Duración de clips (segundos) ────────────────────
  min=0.00s  mean=3.01s  max=11.60s
  Duración total de clips: 21.4 horas

── Por split ────────────────────────────────────────
  train: 16054 clips  1000 clases
  val  :  5287 clips  1000 clases
  test :  4172 clips  1000 clases

── Eficiencia de agrupación ─────────────────────────
  Descargas únicas:  7,213
  Descargas evitadas: 18,300 (71.7%)


In [13]:
# ── Funciones auxiliares ──────────────────────────────────────────────────────

def get_video_dims(video_path: Path) -> tuple[int, int]:
    """Devuelve (width, height) reales del vídeo descargado via ffprobe."""
    result = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-select_streams', 'v:0',
         '-show_entries', 'stream=width,height', '-of', 'csv=p=0', str(video_path)],
        capture_output=True, text=True
    )
    parts = result.stdout.strip().split(',')
    return int(parts[0]), int(parts[1])


def build_vf_filter(box: list, actual_w: int, actual_h: int,
                    margin: float, target_height: int) -> str:
    """
    Construye el filtro -vf de ffmpeg para crop + resize.
    box = [y0, x0, y1, x1] normalizado (0-1).
    """
    y0, x0, y1, x1 = box

    # Ancho y alto del box en píxeles
    bw = (x1 - x0) * actual_w
    bh = (y1 - y0) * actual_h

    # Margen en píxeles
    mx = margin * bw
    my = margin * bh

    # Esquina superior izquierda del crop (con margen, clamped a bordes)
    cx = max(0, int(x0 * actual_w - mx))
    cy = max(0, int(y0 * actual_h - my))

    # Tamaño del crop (con margen, clamped al frame)
    cw = min(actual_w - cx, int(bw + 2 * mx))
    ch = min(actual_h - cy, int(bh + 2 * my))

    # Asegurar dimensiones pares (requerido por libx264)
    cw = cw - (cw % 2)
    ch = ch - (ch % 2)

    vf = f'crop={cw}:{ch}:{cx}:{cy}'
    if target_height > 0:
        vf += f',scale=-2:{target_height}'
    return vf


def classify_yt_error(error_str: str) -> str:
    """Clasifica el error de yt-dlp en una categoría."""
    e = error_str.lower()
    if 'private' in e:                             return 'private'
    if 'age' in e and ('confirm' in e or 'restricted' in e): return 'age_restricted'
    if 'unavailable' in e or 'removed' in e:       return 'deleted'
    if 'not available in your country' in e:       return 'geo_blocked'
    if 'members-only' in e or 'members only' in e: return 'members_only'
    if 'http error 403' in e or 'forbidden' in e:  return 'access_denied'
    if 'network' in e or 'timeout' in e or 'connection' in e: return 'network_error'
    return 'unknown'


def ffmpeg_trim_and_crop(temp_path: Path, out_path: Path,
                          start_time: float, end_time: float,
                          box: list, crop: bool,
                          margin: float, target_height: int) -> bool:
    """
    Recorta el segmento [start_time, end_time] y opcionalmente
    aplica crop al signante. Devuelve True si OK, False si error.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)

    vf = None
    if crop:
        try:
            actual_w, actual_h = get_video_dims(temp_path)
            vf = build_vf_filter(box, actual_w, actual_h, margin, target_height)
        except Exception:
            vf = None  # Si ffprobe falla, procesar sin crop

    duration = end_time - start_time
    cmd = ['ffmpeg', '-y',
           '-ss', str(start_time),
           '-t',  str(max(duration, 0.5)),  # mínimo 0.5s
           '-i',  str(temp_path)]

    if vf:
        cmd += ['-vf', vf]

    cmd += ['-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
            '-an',   # sin audio (no relevante para lengua de signos)
            str(out_path)]

    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0 and out_path.exists() and out_path.stat().st_size > 1000


print('Funciones auxiliares definidas.')

Funciones auxiliares definidas.


In [14]:
# ── Sistema de progreso (resume automático) ───────────────────────────────────

def load_progress() -> tuple[set, dict]:
    """Carga clip_ids completados y errores previos."""
    completed = set()
    errors    = {}
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            data = json.load(f)
        completed = set(data.get('completed_ids', []))
        print(f'Resume: {len(completed):,} clips ya completados.')
    if ERRORS_FILE.exists():
        with open(ERRORS_FILE) as f:
            errors = json.load(f)
        print(f'Resume: {len(errors):,} errores previos cargados.')
    return completed, errors


def save_progress(completed: set, errors: dict):
    """Guardado atómico: escribe a .tmp y renombra (POSIX-safe)."""
    # Progress
    tmp = PROGRESS_FILE.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump({'completed_ids': list(completed),
                   'last_saved': datetime.now().isoformat()}, f)
    os.replace(tmp, PROGRESS_FILE)

    # Errors
    tmp_e = ERRORS_FILE.with_suffix('.tmp')
    with open(tmp_e, 'w') as f:
        json.dump(errors, f, indent=2)
    os.replace(tmp_e, ERRORS_FILE)


completed_ids, error_log = load_progress()

# Filtrar URLs donde ya están todos los clips completados
urls_todo = {}
for url, clips in manifest.items():
    remaining = [c for c in clips if c['clip_id'] not in completed_ids]
    if remaining:
        urls_todo[url] = remaining

print(f'URLs pendientes: {len(urls_todo):,} / {len(manifest):,}')
print(f'Clips pendientes: {sum(len(v) for v in urls_todo.values()):,}')

Resume: 19 clips ya completados.
Resume: 6 errores previos cargados.
URLs pendientes: 7,198 / 7,213
Clips pendientes: 25,494


In [15]:
# ── Función de descarga con yt-dlp ────────────────────────────────────────────

def download_video(url: str, temp_path: Path) -> bool:
    """
    Descarga el vídeo a temp_path usando yt-dlp.
    Devuelve True si OK. El archivo descargado puede tener extensión
    distinta a .mp4, por eso usamos %(ext)s en outtmpl y buscamos el resultado.
    """
    ydl_opts = {
        'format': f'bestvideo[height<={MAX_HEIGHT}][ext=mp4]/bestvideo[height<={MAX_HEIGHT}]/best[height<={MAX_HEIGHT}]',
        'outtmpl': str(temp_path.with_suffix('.%(ext)s')),
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': False,
        'nocheckcertificate': True,
        'socket_timeout': 30,
        'retries': 3,
        'fragment_retries': 3,
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        # Buscar el archivo descargado (puede ser .mp4, .webm, etc.)
        for ext in ['mp4', 'webm', 'mkv', 'avi']:
            candidate = temp_path.with_suffix(f'.{ext}')
            if candidate.exists() and candidate.stat().st_size > 10_000:
                if ext != 'mp4':
                    # Convertir a mp4 para ffmpeg
                    mp4_path = temp_path.with_suffix('.mp4')
                    r = subprocess.run(
                        ['ffmpeg', '-y', '-i', str(candidate),
                         '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
                         '-an', str(mp4_path)],
                        capture_output=True
                    )
                    candidate.unlink(missing_ok=True)
                    return r.returncode == 0
                return True
        return False
    except yt_dlp.utils.DownloadError:
        raise
    except Exception:
        return False


def cleanup_temp(temp_path: Path):
    """Borra todos los archivos temporales de esta URL."""
    for ext in ['mp4', 'webm', 'mkv', 'avi', 'part', 'ytdl']:
        f = temp_path.with_suffix(f'.{ext}')
        if f.exists():
            try: f.unlink()
            except Exception: pass


print('Función de descarga definida.')

Función de descarga definida.


In [16]:
# ── Bucle principal de descarga ───────────────────────────────────────────────
# Este cell puede tardar varias horas. Puedes interrumpir con Kernel > Interrupt
# y reanudar ejecutando desde la cell de 'load_progress' en adelante.

url_list = list(urls_todo.items())
total_urls = len(url_list)
total_clips_pending = sum(len(v) for v in urls_todo.values())

print(f'Iniciando descarga: {total_urls:,} URLs  |  {total_clips_pending:,} clips pendientes')
print(f'Crop: CROP_SIGNER={CROP_SIGNER}  margin={CROP_MARGIN}  target_height={TARGET_HEIGHT}px')
print()

stats = {
    'urls_ok': 0, 'urls_failed': 0,
    'clips_ok': 0, 'clips_failed': 0,
}

t_start = time.time()
urls_since_checkpoint = 0

pbar = tqdm(url_list, desc='URLs', unit='url')

for url_idx, (url, clips) in enumerate(pbar):
    url_hash = hashlib.sha1(url.encode()).hexdigest()[:16]
    temp_path = TEMP_DIR / url_hash  # sin extensión (yt-dlp añade la correcta)

    # ── Intentar descargar el vídeo ───────────────────────────────────────
    download_ok = False
    error_category = None
    error_msg = ''

    for attempt in range(MAX_RETRIES + 1):
        try:
            cleanup_temp(temp_path)
            download_ok = download_video(url, temp_path)
            if download_ok:
                break
        except yt_dlp.utils.DownloadError as e:
            error_msg = str(e)
            error_category = classify_yt_error(error_msg)
            # No reintentar si es error permanente (vídeo borrado, privado…)
            if error_category in ('private', 'deleted', 'age_restricted',
                                  'geo_blocked', 'members_only', 'access_denied'):
                break
        except Exception as e:
            error_msg = str(e)
            error_category = 'unknown'

    # ── Procesar clips ────────────────────────────────────────────────────
    if not download_ok:
        # Marcar todos los clips de esta URL como fallidos
        cat = error_category or 'download_failed'
        for clip in clips:
            error_log[clip['clip_id']] = {
                'url': url, 'label': clip['label'], 'text': clip['text'],
                'split': clip['split'], 'error': cat, 'msg': error_msg[:200],
            }
        stats['urls_failed'] += 1
        stats['clips_failed'] += len(clips)
        pbar.set_postfix(ok=stats['urls_ok'], fail=stats['urls_failed'],
                         clips_ok=stats['clips_ok'])
    else:
        # Encontrar el archivo descargado
        temp_mp4 = temp_path.with_suffix('.mp4')
        url_clips_ok = 0

        for clip in clips:
            out_path = Path(clip['out_path'])
            if clip['clip_id'] in completed_ids:
                url_clips_ok += 1
                continue

            clip_ok = ffmpeg_trim_and_crop(
                temp_mp4, out_path,
                clip['start_time'], clip['end_time'],
                clip['box'], CROP_SIGNER,
                CROP_MARGIN, TARGET_HEIGHT
            )

            if clip_ok:
                completed_ids.add(clip['clip_id'])
                stats['clips_ok'] += 1
                url_clips_ok += 1
            else:
                error_log[clip['clip_id']] = {
                    'url': url, 'label': clip['label'], 'text': clip['text'],
                    'split': clip['split'], 'error': 'ffmpeg_error', 'msg': '',
                }
                stats['clips_failed'] += 1

        stats['urls_ok'] += 1
        pbar.set_postfix(ok=stats['urls_ok'], fail=stats['urls_failed'],
                         clips_ok=stats['clips_ok'])

    # ── Limpiar temp ──────────────────────────────────────────────────────
    cleanup_temp(temp_path)

    # ── Checkpoint ────────────────────────────────────────────────────────
    urls_since_checkpoint += 1
    if urls_since_checkpoint >= CHECKPOINT_EVERY:
        save_progress(completed_ids, error_log)
        urls_since_checkpoint = 0
        elapsed = time.time() - t_start
        done_pct = (url_idx + 1) / total_urls
        eta_s = (elapsed / done_pct) * (1 - done_pct) if done_pct > 0 else 0
        eta_str = str(timedelta(seconds=int(eta_s)))
        tqdm.write(f'[Checkpoint] {url_idx+1}/{total_urls} URLs  '
                   f'clips_ok={stats["clips_ok"]:,}  '
                   f'failed_urls={stats["urls_failed"]:,}  ETA={eta_str}')

# Guardado final
save_progress(completed_ids, error_log)

print()
print('── Descarga completada ─────────────────────────────')
print(f'  URLs OK:      {stats["urls_ok"]:,}')
print(f'  URLs fallidas: {stats["urls_failed"]:,}')
print(f'  Clips OK:     {stats["clips_ok"]:,}')
print(f'  Clips fallidos: {stats["clips_failed"]:,}')
print(f'  Tiempo total:   {str(timedelta(seconds=int(time.time() - t_start)))}')

Iniciando descarga: 7,198 URLs  |  25,494 clips pendientes
Crop: CROP_SIGNER=True  margin=0.12  target_height=360px



URLs:   0%|          | 0/7198 [00:00<?, ?url/s]

ERROR: [youtube] 9FdHlMOnVjg: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] 1AyT77LqJzQ: Video unavailable
ERROR: [youtube] cJOyCgIKyeA: Video unavailable
ERROR: [youtube] zqfcp6NRKpA: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] 7y5Ye-2-ZBs: Private video. Sign in if you've been granted acce

ERROR: [youtube] rYW7VFPi9_w: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


ERROR: [youtube] 0Beq_NIDj2c: Video unavailable


...

[Checkpoint] 7180/7198 URLs  clips_ok=16,781  failed_urls=3,092  ETA=0:00:30


ERROR: [youtube] 6JlqUMRxHwo: Video unavailable
ERROR: [youtube] rVMTe9k9Fks: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] _2YnCKHJy6U: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] gQsIUo46pJU: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or 

ERROR: [youtube] _avAtcE6T_o: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] T8c5eHxZik0: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] 8iu4yAES0Dg: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://g

ERROR: [youtube] -wYQN6l-UJc: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] KJgjS-tHBp0: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] XKsE3bqirCc: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://g


── Descarga completada ─────────────────────────────
  URLs OK:      4,090
  URLs fallidas: 3,108
  Clips OK:     16,797
  Clips fallidos: 8,695
  Tiempo total:   3:23:02


In [17]:
# ── Auditoría post-descarga ───────────────────────────────────────────────────
# Escanea los archivos reales en disco y cruza con el manifiesto.

print('Escaneando archivos en disco...')

audit = {
    'generated_at': datetime.now().isoformat(),
    'total_clips_expected': total_clips,
    'splits': {},
    'error_breakdown': defaultdict(int),
    'per_class_coverage': {},
}

# Contar clips en disco por split
for split_name in ['train', 'val', 'test']:
    split_dir = OUTPUT_DIR / split_name
    if not split_dir.exists():
        audit['splits'][split_name] = {'on_disk': 0, 'expected': 0, 'coverage': 0}
        continue
    on_disk = list(split_dir.rglob('*.mp4'))
    expected = sum(1 for clips in manifest.values()
                   for c in clips if c['split'] == split_name)
    coverage = 100 * len(on_disk) / expected if expected > 0 else 0
    audit['splits'][split_name] = {
        'on_disk': len(on_disk),
        'expected': expected,
        'coverage_pct': round(coverage, 1),
    }

# Desglose de errores por categoría
for clip_id, err in error_log.items():
    audit['error_breakdown'][err['error']] += 1

# Cobertura por clase (top-100 para el subset inicial)
class_clips_expected = defaultdict(int)
class_clips_on_disk  = defaultdict(int)
completed_set = set(completed_ids)

for clips in manifest.values():
    for c in clips:
        class_clips_expected[c['label']] += 1
        if c['clip_id'] in completed_set:
            class_clips_on_disk[c['label']] += 1

for label in range(min(100, len(classes))):
    exp = class_clips_expected.get(label, 0)
    got = class_clips_on_disk.get(label, 0)
    audit['per_class_coverage'][f"{label:04d}_{classes[label]}"] = {
        'expected': exp, 'on_disk': got,
        'coverage_pct': round(100 * got / exp, 1) if exp > 0 else 0
    }

# Métricas globales
total_on_disk = sum(s['on_disk'] for s in audit['splits'].values())
global_coverage = 100 * total_on_disk / total_clips if total_clips > 0 else 0
audit['total_clips_on_disk'] = total_on_disk
audit['global_coverage_pct'] = round(global_coverage, 1)

# Mostrar resumen
print()
print('── Resumen de auditoría ─────────────────────────────')
print(f'  Total esperado:  {total_clips:,}')
print(f'  Total en disco:  {total_on_disk:,}  ({global_coverage:.1f}%)')
print()
for split_name, s in audit['splits'].items():
    print(f'  {split_name:5s}: {s["on_disk"]:5d} / {s["expected"]:5d}  ({s["coverage_pct"]}%)')
print()
print('  Errores por categoría:')
for cat, count in sorted(audit['error_breakdown'].items(), key=lambda x: -x[1]):
    print(f'    {cat:20s}: {count:,}')

# Cobertura media en top-100
top100_coverages = [v['coverage_pct'] for v in audit['per_class_coverage'].values()]
if top100_coverages:
    print(f'\n  Cobertura media top-100 clases: {sum(top100_coverages)/len(top100_coverages):.1f}%')
    zero_coverage = sum(1 for c in top100_coverages if c == 0)
    print(f'  Clases con 0 clips (top-100):   {zero_coverage}')

Escaneando archivos en disco...

── Resumen de auditoría ─────────────────────────────
  Total esperado:  25,513
  Total en disco:  16,816  (65.9%)

  train: 11884 / 16054  (74.0%)
  val  :  2510 /  5287  (47.5%)
  test :  2422 /  4172  (58.1%)

  Errores por categoría:
    private             : 3,519
    deleted             : 3,139
    unknown             : 2,037

  Cobertura media top-100 clases: 63.7%
  Clases con 0 clips (top-100):   0


In [18]:
# ── Guardar audit JSON + missing_videos.log ───────────────────────────────────

# Audit JSON (completo, para Nb 20)
with open(AUDIT_FILE, 'w') as f:
    json.dump(audit, f, indent=2)
print(f'Guardado: {AUDIT_FILE}')

# missing_videos.log (legible por humanos)
with open(LOG_FILE, 'w') as log:
    log.write(f'MS-ASL Missing Videos Log — {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    log.write('=' * 70 + '\n\n')

    # Agrupar errores por categoría
    by_category = defaultdict(list)
    for clip_id, err in error_log.items():
        by_category[err['error']].append(err)

    log.write(f'RESUMEN:\n')
    log.write(f'  Total clips esperados:  {total_clips:,}\n')
    log.write(f'  Total clips en disco:   {total_on_disk:,} ({global_coverage:.1f}%)\n')
    log.write(f'  Total clips fallidos:   {len(error_log):,}\n\n')

    for category, entries in sorted(by_category.items(), key=lambda x: -len(x[1])):
        log.write(f'\n── {category.upper()} ({len(entries)} clips) ──\n')
        # Agrupar por URL para no repetir
        urls_in_cat = {}
        for e in entries:
            if e['url'] not in urls_in_cat:
                urls_in_cat[e['url']] = {'count': 0, 'sample': e['text']}
            urls_in_cat[e['url']]['count'] += 1
        for url, info in list(urls_in_cat.items())[:50]:  # max 50 por categoría
            log.write(f'  [{info["count"]:3d} clips] {url}  (ej: {info["sample"]})\n')
        if len(urls_in_cat) > 50:
            log.write(f'  ... y {len(urls_in_cat) - 50} URLs más\n')

print(f'Guardado: {LOG_FILE}')

# Tamaño total en disco
total_size_mb = sum(
    f.stat().st_size for f in OUTPUT_DIR.rglob('*.mp4') if f.is_file()
) / (1024**2)
print(f'\nTamaño total en disco: {total_size_mb:.0f} MB  ({total_size_mb/1024:.1f} GB)')

Guardado: ../../06_datasets/MS-ASL/videos/_audit.json
Guardado: ../../06_datasets/MS-ASL/videos/missing_videos.log

Tamaño total en disco: 1557 MB  (1.5 GB)


## Resumen

| Métrica | Valor |
|---------|-------|
| URLs únicas procesadas | ~7,213 |
| Clips totales esperados | ~25,513 |
| Clips en disco | ver `_audit.json` |
| Cobertura global | ver `_audit.json` |
| Errores más comunes | ver `missing_videos.log` |

### Archivos generados
- `06_datasets/MS-ASL/videos/{train,val,test}/{label:04d}/{clip_id}.mp4` — clips recortados
- `_progress.json` — IDs completados (para reanudar)
- `_errors.json` — errores con categoría
- `_audit.json` — resumen completo (usado por Nb 20)
- `missing_videos.log` — log legible por humanos

### Siguiente paso
→ **Notebook 20 — MS-ASL Landmark Extraction**: aplicar MediaPipe HandLandmarker
a todos los clips descargados (mismo pipeline que Nb 15 de WLASL).